In [1]:
# PyMC-Marketing = MMM bayesiano em Python, roda em CPU
!pip install -q pymc-marketing
import pymc_marketing
print("versão:", pymc_marketing.__version__)  # anote essa versão para o requirements depois

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.8/519.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.5/273.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.3/514.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
versão: 0.19.3


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_semanas = 150
datas = pd.date_range("2023-01-01", periods=n_semanas, freq="W")

# Gasto semanal em 3 canais (com padrões diferentes, como na vida real)
tv     = np.random.gamma(2, 1, n_semanas) * 1000
social = np.random.gamma(2, 1, n_semanas) * 600
search = np.random.gamma(2, 1, n_semanas) * 400

def adstock(x, taxa):
    """Efeito da mídia que perdura: a propaganda de hoje ainda gera venda amanhã."""
    y = np.zeros_like(x)
    for i in range(len(x)):
        y[i] = x[i] + (taxa * y[i-1] if i > 0 else 0)
    return y

def saturacao(x, k):
    """Retorno decrescente: dobrar o gasto não dobra a venda."""
    return x / (x + k)

# VERDADE que definimos (ROIs e parâmetros). O modelo tem que redescobrir isso.
tv_efeito     = saturacao(adstock(tv, 0.6), 4000) * 5000
social_efeito = saturacao(adstock(social, 0.3), 2000) * 3000
search_efeito = saturacao(adstock(search, 0.1), 1000) * 2500

tendencia = np.linspace(8000, 12000, n_semanas)
sazonal   = 1500 * np.sin(np.arange(n_semanas) * 2 * np.pi / 52)
ruido     = np.random.normal(0, 800, n_semanas)
vendas    = tendencia + sazonal + tv_efeito + social_efeito + search_efeito + ruido

df = pd.DataFrame({"date": datas, "tv": tv, "social": social,
                   "search": search, "vendas": vendas})
print(df.head())
print("\nGround-truth de contribuição total por canal (o que o modelo deve recuperar):")
for nome, ef in [("tv", tv_efeito), ("social", social_efeito), ("search", search_efeito)]:
    print(f"  {nome}: {ef.sum():,.0f} em vendas")

        date           tv       social       search        vendas
0 2023-01-01  2393.679390  1892.777443   410.976763  12402.467095
1 2023-01-08  1494.464730  3709.035585  1137.777521  13884.806286
2 2023-01-15  1382.283584  2031.885830   719.288483  13839.400835
3 2023-01-22  1382.302294  2337.360632  1134.827130  13086.590981
4 2023-01-29  4649.714412   545.900457   379.693263  13197.305531

Ground-truth de contribuição total por canal (o que o modelo deve recuperar):
  tv: 397,197 em vendas
  social: 198,301 em vendas
  search: 157,885 em vendas


In [3]:
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation

X = df[["date", "tv", "social", "search"]]
y = df["vendas"]

# adstock = efeito que perdura (l_max = quantas semanas de memória)
# saturação logística = retornos decrescentes por canal
mmm = MMM(
    date_column="date",
    channel_columns=["tv", "social", "search"],
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    yearly_seasonality=2,
)

# Esta é a linha que mais varia por versão. Se der erro de assinatura,
# confira o quickstart oficial e ajuste (alguns usam mmm.fit(X, y); outros pedem kwargs).
mmm.fit(X, y, chains=4, draws=1000, tune=1000, random_seed=42)

print("Modelo treinado. Diagnóstico de convergência:")
import arviz as az
print(az.summary(mmm.fit_result, var_names=["~mu"], round_to=2).head(15))

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_validate_call.py:136: FutureWarning: 
            The MMM class is deprecated and will be removed in a future version (in version 0.20.0).
            Please use the multidimensional MMM class instead.
            That is, `from pymc_marketing.mmm.multidimensional import MMM`.
            All our documentation has been updated to reflect this change.
            Refer to the migration guide for more details: https://www.pymc-marketing.io/en/latest/notebooks/mmm/mmm_migration_guide.html
            
  res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))


Output()

ERROR:pymc.stats.convergence:There were 71 divergences after tuning. Increase `target_accept` or reparameterize.


Output()

Modelo treinado. Diagnóstico de convergência:
                         mean    sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
adstock_alpha[tv]        0.71  0.16    0.40     0.94       0.00     0.00   
adstock_alpha[social]    0.20  0.17    0.00     0.51       0.00     0.00   
adstock_alpha[search]    0.16  0.13    0.00     0.38       0.00     0.00   
gamma_fourier[sin_1]     0.06  0.01    0.04     0.08       0.00     0.00   
gamma_fourier[sin_2]    -0.01  0.01   -0.02     0.01       0.00     0.00   
gamma_fourier[cos_1]    -0.01  0.01   -0.02     0.01       0.00     0.00   
gamma_fourier[cos_2]    -0.01  0.01   -0.02     0.01       0.00     0.00   
intercept                0.66  0.04    0.59     0.75       0.00     0.00   
saturation_beta[tv]      0.53  0.47    0.02     1.32       0.01     0.03   
saturation_beta[social]  0.09  0.13    0.00     0.24       0.00     0.01   
saturation_beta[search]  0.16  0.17    0.02     0.35       0.01     0.02   
saturation_lam[tv]       2.21  1.45    0.2

In [4]:
# Contribuição estimada por canal vs a verdade que definimos na Fase 1
contrib = mmm.compute_channel_contribution_original_scale()
estimado = contrib.mean(dim=["chain", "draw"]).sum(dim="date").to_series()

print("Canal   | Estimado pelo modelo | Verdade real")
for canal, real in [("tv", tv_efeito.sum()), ("social", social_efeito.sum()),
                    ("search", search_efeito.sum())]:
    est = estimado.get(canal, float("nan"))
    erro = 100 * (est - real) / real
    print(f"{canal:7s} | {est:>18,.0f} | {real:>12,.0f}  ({erro:+.0f}%)")

Canal   | Estimado pelo modelo | Verdade real
tv      |            267,185 |      397,197  (-33%)
social  |             46,539 |      198,301  (-77%)
search  |            118,535 |      157,885  (-25%)


In [6]:
contrib = mmm.compute_channel_contribution_original_scale()

In [7]:
import json
import numpy as np

# Para cada canal, montamos uma tabela: gasto -> venda prevista.
# Usamos a contribuição já estimada (contrib), sem depender de método interno
# da biblioteca, que muda de assinatura entre versões.
grid = 60
curvas = {}

contrib_media = contrib.mean(dim=["chain", "draw"])  # média da posterior por canal/semana

for canal in ["tv", "social", "search"]:
    gasto_max = df[canal].max() * 1.5
    eixo_gasto = np.linspace(0, gasto_max, grid)

    # pares (gasto observado, contribuição estimada) para esse canal, ordenados
    gasto_obs = df[canal].values
    contrib_obs = contrib_media.sel(channel=canal).values
    ordem = np.argsort(gasto_obs)

    # curva gasto -> venda: interpolamos sobre os pontos observados ordenados
    venda = np.interp(eixo_gasto, gasto_obs[ordem], contrib_obs[ordem])

    curvas[canal] = {
        "gasto": eixo_gasto.tolist(),
        "venda": np.maximum(venda, 0).tolist(),  # venda não pode ser negativa
    }

with open("curvas_resposta.json", "w") as f:
    json.dump(curvas, f)

print("Curvas salvas com sucesso!")
for canal in curvas:
    vmax = max(curvas[canal]["venda"])
    print(f"  {canal}: venda máxima prevista ~ {vmax:,.0f}")

# baixa o artefato para o PC
from google.colab import files
files.download("curvas_resposta.json")

Curvas salvas com sucesso!
  tv: venda máxima prevista ~ 2,924
  social: venda máxima prevista ~ 689
  search: venda máxima prevista ~ 1,845


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
print([m for m in dir(mmm) if "contrib" in m.lower()])

['_build_channel_contribution', '_build_control_contribution', '_build_yearly_seasonality_contribution', '_format_model_contributions', 'channel_contribution_forward_pass', 'compute_channel_contribution_original_scale', 'compute_mean_contributions_over_time', 'get_channel_contribution_forward_pass_grid', 'get_channel_contribution_share_samples', 'get_ts_contribution_posterior', 'new_spend_contributions', 'plot_allocated_contribution_by_channel', 'plot_channel_contribution_grid', 'plot_channel_contribution_share_hdi', 'plot_components_contributions', 'plot_direct_contribution_curves', 'plot_grouped_contribution_breakdown_over_time', 'plot_new_spend_contributions']


In [9]:
import json
import numpy as np
from scipy.optimize import curve_fit

# Curva de saturação de Hill: monotônica e côncava, a forma "certa" de resposta de mídia
def hill(x, vmax, k):
    return vmax * x / (k + x)

grid = 80
curvas = {}

# usamos a posterior inteira para extrair média E incerteza
contrib_todas = contrib  # dims: chain, draw, date, channel
contrib_media = contrib_todas.mean(dim=["chain", "draw"])

for canal in ["tv", "social", "search"]:
    gasto_obs = df[canal].values
    contrib_obs = contrib_media.sel(channel=canal).values
    gmax = gasto_obs.max() * 1.5
    eixo = np.linspace(0, gmax, grid)

    # ajusta a curva de Hill aos pontos observados (suaviza o ruído)
    p0 = [contrib_obs.max(), np.median(gasto_obs)]
    try:
        (vmax, k), _ = curve_fit(hill, gasto_obs, contrib_obs, p0=p0, maxfev=10000)
        venda = hill(eixo, vmax, k)
    except Exception:
        venda = np.interp(eixo, np.sort(gasto_obs), np.sort(contrib_obs))

    # banda de incerteza: dispersão da posterior, escalada ao longo da curva
    contrib_std = contrib_todas.sel(channel=canal).std(dim=["chain", "draw"]).mean().item()
    venda = np.maximum(venda, 0)
    curvas[canal] = {
        "gasto": eixo.tolist(),
        "venda": venda.tolist(),
        "venda_low":  np.maximum(venda - contrib_std, 0).tolist(),
        "venda_high": (venda + contrib_std).tolist(),
    }

with open("curvas_resposta.json", "w") as f:
    json.dump(curvas, f)

print("Curvas suavizadas salvas!")
for c in curvas:
    print(f"  {c}: venda máxima ~ {max(curvas[c]['venda']):,.0f}")

from google.colab import files
files.download("curvas_resposta.json")

Curvas suavizadas salvas!
  tv: venda máxima ~ 2,105
  social: venda máxima ~ 785
  search: venda máxima ~ 2,022


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>